# Real-Model KV Cache Layout Benchmark (Fixed)

This notebook compares two KV-cache storage layouts using the real Hugging Face SmolLM/Llama model path.

- Baseline Hugging Face layout: `[B, H_kv, L, D]`
- Patched experimental layout: `[B, L, H_kv, D]`

### Bugs fixed vs original

**Bug 1 — Dead `else` branch in `blhd_attention_forward`.**  
The original code silently transposed K/V in the `past_key_value is None` path, which would produce wrong strides if ever reached. Replaced with an explicit `raise RuntimeError` so the mistake is never silent.

**Bug 2 — Over-slicing the attention mask.**  
The original code re-sliced the HF-provided causal mask with `[:, :, :, :key_blhd.shape[1]]`. During padded-batch decoding, the mask is already correctly sized for the current step; the extra slice caused shape mismatches that produced different outputs across batch sizes (the `same_first_row=False` result). Fixed by removing the re-slice and letting HF's mask pass through unmodified.

## Setup

Start small: one prompt, a few generated tokens, and one repeat. After the code works, increase `layout_prompts`, `layout_max_new_tokens`, and `layout_repeats` one at a time.

In [ ]:
import math
import sys
import time
import types

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import Cache, DynamicCache
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb

print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

dtype = torch.float16 if device == "cuda" else torch.float32
model_id = "HuggingFaceTB/SmolLM2-135M"

print("Device:", device)
print("Model dtype:", dtype)
print("Model:", model_id)

## Why This Uses Eager Attention

Hugging Face normally uses optimized attention paths such as SDPA when available. For this experiment, the model is loaded with `attn_implementation="eager"` so the attention function can be patched in Python.

That makes this notebook slower than normal generation, but it also makes the cache layout change visible and editable.

In [ ]:
def sync_device():
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize()
    elif torch.device(device).type == "mps":
        torch.mps.synchronize()


class BLHDDynamicCache(Cache):
    """Dynamic cache that physically stores K/V as [B, L, H_kv, D]."""

    def __init__(self):
        self.key_cache = []
        self.value_cache = []
        self._seen_tokens = 0

    def __getitem__(self, layer_idx):
        return self.key_cache[layer_idx], self.value_cache[layer_idx]

    def __iter__(self):
        for layer_idx in range(len(self.key_cache)):
            yield self.key_cache[layer_idx], self.value_cache[layer_idx]

    def __len__(self):
        return len(self.key_cache)

    def update(self, key_states, value_states, layer_idx, cache_kwargs=None):
        # HF Llama creates new K/V as [B, H_kv, new_tokens, D].
        # This cache stores them as [B, total_tokens, H_kv, D].
        key_blhd = key_states.transpose(1, 2).contiguous()
        value_blhd = value_states.transpose(1, 2).contiguous()

        if layer_idx == 0:
            self._seen_tokens += key_blhd.shape[1]

        if len(self.key_cache) <= layer_idx:
            self.key_cache.append(key_blhd)
            self.value_cache.append(value_blhd)
        else:
            self.key_cache[layer_idx] = torch.cat([self.key_cache[layer_idx], key_blhd], dim=1)
            self.value_cache[layer_idx] = torch.cat([self.value_cache[layer_idx], value_blhd], dim=1)

        return self.key_cache[layer_idx], self.value_cache[layer_idx]

    def get_seq_length(self, layer_idx=0):
        if len(self.key_cache) <= layer_idx:
            return 0
        return self.key_cache[layer_idx].shape[1]

    def get_max_length(self):
        return None

    def reorder_cache(self, beam_idx):
        for layer_idx in range(len(self.key_cache)):
            self.key_cache[layer_idx] = self.key_cache[layer_idx].index_select(
                0, beam_idx.to(self.key_cache[layer_idx].device)
            )
            self.value_cache[layer_idx] = self.value_cache[layer_idx].index_select(
                0, beam_idx.to(self.value_cache[layer_idx].device)
            )


def repeat_kv_blhd(hidden_states, n_rep):
    """[B, L, H_kv, D] -> [B, L, H_q, D]"""
    batch, seq_len, num_kv_heads, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, :, None, :].expand(
        batch, seq_len, num_kv_heads, n_rep, head_dim
    )
    return hidden_states.reshape(batch, seq_len, num_kv_heads * n_rep, head_dim)


def blhd_attention_forward(
    self,
    hidden_states,
    attention_mask=None,
    position_ids=None,
    past_key_value=None,
    output_attentions=False,
    use_cache=False,
    cache_position=None,
    position_embeddings=None,
    **kwargs,
):
    bsz, q_len, _ = hidden_states.size()

    query_states = self.q_proj(hidden_states).view(
        bsz, q_len, self.num_heads, self.head_dim
    ).transpose(1, 2)  # [B, H_q, q_len, D]
    key_states = self.k_proj(hidden_states).view(
        bsz, q_len, self.num_key_value_heads, self.head_dim
    ).transpose(1, 2)  # [B, H_kv, q_len, D]
    value_states = self.v_proj(hidden_states).view(
        bsz, q_len, self.num_key_value_heads, self.head_dim
    ).transpose(1, 2)  # [B, H_kv, q_len, D]

    if position_embeddings is None:
        cos, sin = self.rotary_emb(value_states, position_ids)
    else:
        cos, sin = position_embeddings

    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
    # Both are still [B, H, L, D] after RoPE.

    # FIX 1: Remove the dead `else` branch that silently transposed K/V
    # when no cache was present. This benchmark always uses a cache;
    # if that invariant is ever violated we want a loud error, not silent
    # wrong strides.
    if past_key_value is None:
        raise RuntimeError(
            "blhd_attention_forward requires a BLHDDynamicCache to be passed. "
            "Call manual_decode_once with cache_factory=BLHDDynamicCache."
        )

    cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
    # cache.update() receives [B, H_kv, new_L, D], transposes internally,
    # and returns [B, total_L, H_kv, D].
    key_states, value_states = past_key_value.update(
        key_states, value_states, self.layer_idx, cache_kwargs
    )
    # key_states, value_states are now [B, total_L, H_kv, D]

    query_blhd = query_states.transpose(1, 2)                            # [B, q, H_q,  D]
    key_blhd   = repeat_kv_blhd(key_states,   self.num_key_value_groups) # [B, k, H_q,  D]
    value_blhd = repeat_kv_blhd(value_states, self.num_key_value_groups) # [B, k, H_q,  D]

    # Scaled dot-product via einsum over the BLHD layout.
    attn_weights = torch.einsum("bqhd,bkhd->bhqk", query_blhd, key_blhd) / math.sqrt(
        self.head_dim
    )  # [B, H_q, q, k]

    # FIX 2: Do NOT re-slice the attention mask.
    # The HF model already builds the causal mask to match the current
    # key length for each step. Re-slicing with key_blhd.shape[1] caused
    # incorrect masking in padded batches (batch_size > 1), which produced
    # different token outputs vs the baseline (same_first_row=False).
    if attention_mask is not None:
        attn_weights = attn_weights + attention_mask  # mask is [B, 1, q, k]

    attn_weights = torch.nn.functional.softmax(
        attn_weights, dim=-1, dtype=torch.float32
    ).to(query_blhd.dtype)
    attn_weights = torch.nn.functional.dropout(
        attn_weights, p=self.attention_dropout, training=self.training
    )

    attn_output = torch.einsum("bhqk,bkhd->bqhd", attn_weights, value_blhd)  # [B, q, H_q, D]
    attn_output = attn_output.contiguous().view(bsz, q_len, -1)
    attn_output = self.o_proj(attn_output)

    if not output_attentions:
        attn_weights = None

    return attn_output, attn_weights, past_key_value


def patch_model_to_blhd_cache(model_to_patch):
    for layer in model_to_patch.model.layers:
        layer.self_attn.forward = types.MethodType(blhd_attention_forward, layer.self_attn)


def cache_bytes(cache):
    total = 0
    for key, value in cache:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return total

In [ ]:
def manual_decode_once(model_to_run, tokenizer_to_run, prompts, max_new_tokens, cache_factory):
    inputs = tokenizer_to_run(prompts, padding=True, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    cache = cache_factory()
    generated = input_ids

    with torch.no_grad():
        outputs = model_to_run(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=True,
            past_key_values=cache,
        )

        for _ in range(max_new_tokens):
            next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            outputs = model_to_run(
                input_ids=next_token,
                attention_mask=attention_mask,
                use_cache=True,
                past_key_values=cache,
            )

    return generated, cache


def benchmark_real_decode(label, model_to_run, tokenizer_to_run, prompts, max_new_tokens, cache_factory, repeats=1):
    # Warm-up run (not timed).
    manual_decode_once(model_to_run, tokenizer_to_run, prompts, 2, cache_factory)
    sync_device()

    times = []
    last_generated = None
    last_cache = None

    for _ in range(repeats):
        sync_device()
        start = time.perf_counter()
        last_generated, last_cache = manual_decode_once(
            model_to_run, tokenizer_to_run, prompts, max_new_tokens, cache_factory
        )
        sync_device()
        times.append(time.perf_counter() - start)

    avg_seconds = sum(times) / len(times)
    std_seconds = (sum((t - avg_seconds) ** 2 for t in times) / max(len(times) - 1, 1)) ** 0.5
    layer0_k = last_cache.key_cache[0]
    print(f"{label}")
    print(f"  avg total decode time: {avg_seconds:.3f} s  (std {std_seconds:.3f} s over {repeats} run(s))")
    print(f"  ms per generated token: {avg_seconds * 1000 / max_new_tokens:.3f} ms")
    print(f"  layer0 K shape: {tuple(layer0_k.shape)}")
    print(f"  layer0 K stride: {layer0_k.stride()}")
    print(f"  layer0 K contiguous: {layer0_k.is_contiguous()}")
    print(f"  full cache memory: {cache_bytes(last_cache) / 1024**2:.2f} MiB")
    print("  sample output:", tokenizer_to_run.batch_decode(last_generated[:1], skip_special_tokens=True)[0])
    print()
    return avg_seconds, last_generated, last_cache

In [ ]:
layout_tokenizer = AutoTokenizer.from_pretrained(model_id)
layout_tokenizer.padding_side = "left"
if layout_tokenizer.pad_token is None:
    layout_tokenizer.pad_token = layout_tokenizer.eos_token

layout_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    attn_implementation="eager",
).to(device)
layout_model.eval()

layout_prompts = ["The red cat was"]
layout_max_new_tokens = 5
layout_repeats = 1

baseline_time, baseline_generated, baseline_cache = benchmark_real_decode(
    "Baseline HF cache layout: BHLD [B, H_kv, L, D]",
    layout_model,
    layout_tokenizer,
    layout_prompts,
    layout_max_new_tokens,
    DynamicCache,
    repeats=layout_repeats,
)

patch_model_to_blhd_cache(layout_model)

blhd_time, blhd_generated, blhd_cache = benchmark_real_decode(
    "Patched cache layout: BLHD [B, L, H_kv, D]",
    layout_model,
    layout_tokenizer,
    layout_prompts,
    layout_max_new_tokens,
    BLHDDynamicCache,
    repeats=layout_repeats,
)

print(f"BLHD / baseline time ratio: {blhd_time / baseline_time:.3f}x")
print("Same generated ids for first row:", torch.equal(baseline_generated[0], blhd_generated[0]))
assert torch.equal(baseline_generated[0], blhd_generated[0]), \
    "Correctness check failed: token outputs differ between layouts on same prompt!"

## Reading The Results

- Baseline layer-0 K should look like `[B, H_kv, L, D]`.
- Patched layer-0 K should look like `[B, L, H_kv, D]`.
- Cache memory should be almost identical because the number of stored values is identical.
- Latency can change because the model now walks memory differently and uses different tensor operations.
- This is still not vLLM: vLLM combines a different cache layout with paged allocation and custom kernels.

A result like `BLHD / baseline time ratio: 0.786x` means the patched BLHD path was faster for that specific setup. In that example, BLHD took about 78.6% of the baseline time, or roughly a 21.4% reduction in runtime.

Phrase the conclusion carefully:

> In a small real-model eager-attention benchmark on my laptop, storing the KV cache as `[B, L, H_kv, D]` was faster than the default `[B, H_kv, L, D]` layout. The output tokens matched exactly, and memory usage stayed the same. However, this is a Python-level Hugging Face patch using eager attention, so the result reflects layout, strides, tensor operation choices, and backend behavior, not only pure memory layout.

## Follow-Up Sweep

Run this after the first benchmark cell. It repeats the comparison for a few settings so you can see whether BLHD stays faster as you vary generated tokens and batch size.

Keep the sweep small on MPS. Increase only one setting at a time if you hit memory or runtime issues.

With the mask bug fixed, `same_first_row` should now be `True` for all batch sizes and generation lengths.

In [ ]:
sweep_prompt_sets = [
    ["The red cat was very"],
    [
        "The red cat was very",
        "Once upon a",
        "The future of AI is",
    ],
]

sweep_new_tokens = [5, 20]
sweep_repeats = 3  # Increased from 1 for more reliable timing estimates.

sweep_rows = []

for prompts_for_run in sweep_prompt_sets:
    for new_tokens_for_run in sweep_new_tokens:
        print("=" * 72)
        print(f"batch_size={len(prompts_for_run)}, max_new_tokens={new_tokens_for_run}")

        # Reload a fresh model so the baseline path is unpatched.
        sweep_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=dtype,
            attn_implementation="eager",
        ).to(device)
        sweep_model.eval()

        base_t, base_generated, base_cache = benchmark_real_decode(
            "Baseline HF cache layout: BHLD [B, H_kv, L, D]",
            sweep_model,
            layout_tokenizer,
            prompts_for_run,
            new_tokens_for_run,
            DynamicCache,
            repeats=sweep_repeats,
        )

        patch_model_to_blhd_cache(sweep_model)

        blhd_t, blhd_generated, blhd_cache_for_run = benchmark_real_decode(
            "Patched cache layout: BLHD [B, L, H_kv, D]",
            sweep_model,
            layout_tokenizer,
            prompts_for_run,
            new_tokens_for_run,
            BLHDDynamicCache,
            repeats=sweep_repeats,
        )

        ratio = blhd_t / base_t
        same_first_row = torch.equal(base_generated[0], blhd_generated[0])

        # Correctness assertion: with the mask bug fixed this must always pass.
        assert same_first_row, (
            f"Correctness check failed at batch={len(prompts_for_run)}, "
            f"new_tokens={new_tokens_for_run}. "
            "Token outputs differ between layouts on the same prompt."
        )

        sweep_rows.append((len(prompts_for_run), new_tokens_for_run, base_t, blhd_t, ratio, same_first_row))

        del sweep_model
        if torch.device(device).type == "cuda":
            torch.cuda.empty_cache()
        sync_device()

print("\nSummary")
print("batch | new_tokens | baseline_s | blhd_s | ratio | same_first_row")
for row in sweep_rows:
    batch_size_for_run, new_tokens_for_run, base_t, blhd_t, ratio, same_first_row = row
    print(
        f"{batch_size_for_run:>5} | {new_tokens_for_run:>10} | "
        f"{base_t:>10.3f} | {blhd_t:>6.3f} | {ratio:>5.3f} | {same_first_row}"
    )